# Per-class label records


**What this reads (instruction 64.10).** The shipped track record holds one row per held-out gene for each method and label (the gene and its orthogroup hidden, five folds). Every number below is computed from those rows by `parascape.label_metrics`: coverage (share of genes answered), accuracy on answered and on all genes, chance (the commonest class), macro F1, and per class recall, precision and F1. An abstention counts against recall; a class never predicted has no precision (unknown, not zero).

Built 2026-11-07.

In [1]:
import os, pandas as pd, numpy as np
from parascape import label_metrics as LM, track_record as T, strategies as S
ledger = T.shipped()
together = ledger[ledger['mode'].astype(str) == 'together']
pairs = together[['organism', 'strategy', 'target']].astype(str).drop_duplicates().sort_values(['organism', 'target', 'strategy'])
print(len(ledger), 'rows;', len(pairs), 'method x label records')

1038372 rows; 168 method x label records


## 1 · Every method on every label

In [2]:
overall, classes, confusions = [], [], {}
for org, key, target in pairs.itertuples(index=False):
    rows = LM.track_record_rows(together[together.organism.astype(str) == org], key, target)
    o = LM.overall(rows); o.update(organism=org, strategy=key, target=target); overall.append(o)
    per = LM.by_class(rows).assign(organism=org, strategy=key, target=target); classes.append(per)
    confusions[(org, key, target)] = LM.confusion(rows)
overall = pd.DataFrame(overall)[['organism', 'target', 'strategy', 'genes', 'coverage', 'accuracy_answered', 'accuracy_all', 'chance', 'macro_f1', 'classes']]
classes = pd.concat(classes, ignore_index=True)
overall.to_csv(os.path.join(OUT, 'overall.tsv'), sep='\t', index=False)
classes.to_csv(os.path.join(OUT, 'by_class.tsv'), sep='\t', index=False)
with open(os.path.join(OUT, 'confusion.tsv'), 'w') as fh:
    for (org, key, target), table in confusions.items():
        fh.write(f'# {org} {target} {key}\n'); table.to_csv(fh, sep='\t'); fh.write('\n')
overall.assign(beats_chance=overall.accuracy_all > overall.chance).groupby(['organism', 'target']).agg(
    methods=('strategy', 'size'), best_macro_f1=('macro_f1', 'max'), beat_chance=('beats_chance', 'sum')).round(3)

methods  best_macro_f1  beat_chance
organism target                                                           
Pf       is_exported                        14          0.687            0
         lopit_pf_location                  14          0.600           10
         pb_transferred_phenotype           14          0.562            9
Tg       cellcycle_phase                    14          0.576           11
         compartment                        14          0.450            9
         dtm_class                          14          0.628            6
         lopit_unified                      14          0.628            9
         screen_actin_phenotype             14          0.495            0
         screen_any_phenotype               14          0.478            0
         screen_apicoplast_phenotype        14          0.472            0
         screen_egress_phenotype            14          0.527            0
         screen_replication_phenotype       14          0.507            0

## 2 · The best method per label, and its weakest classes

In [3]:
best = overall.loc[overall.groupby(['organism', 'target']).macro_f1.idxmax()]
weak = []
for r in best.itertuples(index=False):
    c = classes[(classes.organism == r.organism) & (classes.strategy == r.strategy) & (classes.target == r.target) & (classes.genes > 0)]
    w = c.nsmallest(2, 'recall')
    weak.append(dict(organism=r.organism, target=r.target, method=r.strategy, macro_f1=round(r.macro_f1, 3),
                     weakest=', '.join(f'{x.label} (recall {x.recall:.0%}, n={x.genes})' for x in w.itertuples(index=False))))
pd.DataFrame(weak)

,organism,target,method,macro_f1,weakest
0,Pf,is_exported,graph_convolution,0.687,"True (recall 73%, n=191), False (recall 94%, n=5529)"
1,Pf,lopit_pf_location,random_forest,0.6,"transport vesicles (recall 13%, n=114), plasma membrane 1 (recall 23%, n=60)"
2,Pf,pb_transferred_phenotype,stacking,0.562,"Slow (recall 36%, n=428), Dispensable (recall 66%, n=832)"
3,Tg,cellcycle_phase,graph_convolution,0.576,"G1a (recall 39%, n=44), S (recall 57%, n=237)"
4,Tg,compartment,stacking,0.45,"apical 2 (recall 18%, n=17), Golgi (recall 29%, n=153)"
5,Tg,dtm_class,random_forest,0.628,"SP+TM (recall 13%, n=266), SP (recall 30%, n=676)"
6,Tg,lopit_unified,stacking,0.628,"pellicle_PM_IMC (recall 48%, n=324), golgi_secretory_vesicular (recall 55%, n=187)"
7,Tg,screen_actin_phenotype,supervised_classifier,0.495,"True (recall 27%, n=44), False (recall 74%, n=274)"
8,Tg,screen_any_phenotype,layer_vote,0.478,"True (recall 27%, n=99), False (recall 62%, n=219)"
9,Tg,screen_apicoplast_phenotype,graph_convolution,0.472,"True (recall 28%, n=60), False (recall 67%, n=258)"


## 3 · Classes the record could not evaluate

Classes with fewer genes than the minimum class size (`strategies.MIN_CLASS`) are left out of every hold-out run; they are evaluation gaps, listed here per label.

In [4]:
gaps = []
for org in together.organism.astype(str).unique():
    ctx = S.Context.shipped(org)
    for target in together[together.organism.astype(str) == org].target.astype(str).unique():
        counts = ctx.truth(target).value_counts()
        small = counts[counts < S.MIN_CLASS]
        gaps.append(dict(organism=org, target=target, min_class=S.MIN_CLASS, classes_left_out=len(small),
                         genes_left_out=int(small.sum()), which=', '.join(f'{k} ({v})' for k, v in small.head(6).items())))
gaps = pd.DataFrame(gaps); gaps.to_csv(os.path.join(OUT, 'evaluation_gaps.tsv'), sep='\t', index=False)
gaps

,organism,target,min_class,classes_left_out,genes_left_out,which
0,Tg,compartment,15,2,23,"20S proteasome (14), tubulin cytoskeleton (9)"
1,Tg,lopit_unified,15,1,9,cytoskeleton (9)
2,Tg,dtm_class,15,1,3,BETA (3)
3,Tg,screen_any_phenotype,15,0,0,
4,Tg,cellcycle_phase,15,0,0,
5,Tg,screen_actin_phenotype,15,0,0,
6,Tg,screen_apicoplast_phenotype,15,0,0,
7,Tg,screen_egress_phenotype,15,0,0,
8,Tg,screen_replication_phenotype,15,0,0,
9,Pf,lopit_pf_location,15,6,66,"19s proteasome (14), ER 2 (13), micronemes 1 (12), micronemes 2 (10), nucleus - histones (9), plasma membrane 2 (8)"
